In [19]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LassoCV

In [20]:
def compute_of_at_level(df, level):
    """
    Compute order flow imbalance (OFI) at a specific book depth level.
    Uses bid/ask price and size deltas based on Cont et al. (2014) method.
    """
    lvl = f"{level:02d}"  # '00' to '09'

    required_columns = [f'bid_px_{lvl}', f'ask_px_{lvl}', f'bid_sz_{lvl}', f'ask_sz_{lvl}']
    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"Column {col} not found in DataFrame.")

    bid_px = df[f'bid_px_{lvl}']
    ask_px = df[f'ask_px_{lvl}']
    bid_sz = df[f'bid_sz_{lvl}']
    ask_sz = df[f'ask_sz_{lvl}']

    # Bid side
    bid_px_diff = bid_px.diff()
    bid_sz_diff = bid_sz.diff()
    of_bid = np.where(
        bid_px_diff > 0,
        bid_sz,
        np.where(bid_px_diff < 0, -bid_sz, bid_sz_diff)
    )

    # Ask side
    ask_px_diff = ask_px.diff()
    ask_sz_diff = ask_sz.diff()
    of_ask = np.where(
        ask_px_diff > 0,
        -ask_sz,
        np.where(ask_px_diff < 0, ask_sz, ask_sz_diff)
    )

    return of_bid, of_ask


In [ ]:
def compute_best_level_ofi(df):
    """
    Compute OFI using only best level (level 0).
    """
    of_bid, of_ask = compute_of_at_level(df, 0)
    return pd.Series(of_bid - of_ask, name='ofi_best_level')

def compute_multi_level_ofi(df, levels=10):
    """
    Compute OFI for each level from 0 to (levels-1), default 10.
    """
    multi_ofi = {}
    for lvl in range(levels):
        of_bid, of_ask = compute_of_at_level(df, lvl)
        multi_ofi[f'ofi_lvl_{lvl:02d}'] = of_bid - of_ask
    return pd.DataFrame(multi_ofi)

def compute_integrated_ofi(multi_ofi_df):
    """
    Use PCA to aggregate multi-level OFIs into one integrated signal.
    """
    norm_df = multi_ofi_df.apply(lambda x: (x - x.mean()) / x.std(), axis=0).fillna(0)
    pca = PCA(n_components=1)
    pc1 = pca.fit_transform(norm_df)
    return pd.Series(pc1.flatten(), name='ofi_integrated')

def compute_cross_asset_ofi(ofi_matrix, target_idx):
    """
    Optional: Estimate cross-impact using LASSO for a target asset vs others.
    """
    y = ofi_matrix.iloc[:, target_idx]
    X = ofi_matrix.drop(ofi_matrix.columns[target_idx], axis=1)
    model = LassoCV(cv=5).fit(X, y)
    return model.coef_, model.intercept_

In [22]:
def main():
    # Load CSV
    try:
        df = pd.read_csv("first_25000_rows.csv")
    except FileNotFoundError:
        raise FileNotFoundError("Cannot find 'first_25000_rows.csv' in current directory.")

    for col in ['ts_recv', 'symbol']:
        if col not in df.columns:
            raise ValueError(f"Required column '{col}' not found in dataset.")

    print("Computing best-level OFI...")
    ofi_best = compute_best_level_ofi(df)

    print("Computing multi-level OFIs...")
    ofi_multi = compute_multi_level_ofi(df)

    print("Computing integrated OFI using PCA...")
    ofi_integrated = compute_integrated_ofi(ofi_multi)

    print("Combining all features...")
    output_df = pd.concat([df[['ts_recv', 'symbol']], ofi_best, ofi_multi, ofi_integrated], axis=1)

    print("Saving to 'ofi_features_output.csv'...")
    output_df.to_csv("ofi_features_output.csv", index=False)

    print("Preview:")
    print(output_df.head())

    
    
    
if __name__ == "__main__":
    main()

Computing best-level OFI...
Computing multi-level OFIs...
Computing integrated OFI using PCA...
Combining all features...
Saving to 'ofi_features_output.csv'...
Preview:
                          ts_recv symbol  ofi_best_level  ofi_lvl_00  \
0  2024-10-21T11:54:29.221230963Z   AAPL             NaN         NaN   
1  2024-10-21T11:54:29.223936626Z   AAPL             2.0         2.0   
2  2024-10-21T11:54:29.225196809Z   AAPL             3.0         3.0   
3  2024-10-21T11:54:29.712600612Z   AAPL             0.0         0.0   
4  2024-10-21T11:54:29.764839221Z   AAPL             0.0         0.0   

   ofi_lvl_01  ofi_lvl_02  ofi_lvl_03  ofi_lvl_04  ofi_lvl_05  ofi_lvl_06  \
0         NaN         NaN         NaN         NaN         NaN         NaN   
1         0.0         0.0         0.0         0.0         0.0         0.0   
2         0.0         0.0         0.0         0.0         0.0         0.0   
3         0.0       200.0         0.0         0.0         0.0         0.0   
4         0.